# S2 J5/J6 — Pipeline wolof de bout en bout

Assemble la chaîne complète en `lang="wo"` : `ffmpeg → transcribe (ASR) → retrieve (RAG) → frontend → synthesize (TTS)`.

**Principe : importer et instrumenter, ne pas réimplémenter.** Chaque brique est testée dans sa propre cellule, un seul modèle en mémoire à la fois — indispensable en local (8 Go RAM ne tiennent pas plusieurs gros modèles ensemble), confortable sur Colab.

**NLU désactivé** (`nlu.enabled: false`) : intent non utilisé, modèle Rasa entraîné en FR. Pas de Rasa à lancer.

## État des maillons

| Maillon | État | Détail |
|---|---|---|
| ASR (Whosper) | ✅ validé | transcrit correctement du wolof (§3) |
| Retrieval (hybride) | ✅ validé | la bonne fiche remonte à partir de la transcription (§4) |
| Génération | ⚠️ extraction directe | Llama 3.2 3B écarté (bascule en indonésien / répétition) ; Oolel-v0.1 bloqué sur la VRAM disponible, reporté S3 (§5bis) |
| Frontend + TTS (Kiriku) | ✅ validé | audio intelligible produit (§6) |
| Latence bout en bout | à mesurer | Colab GPU uniquement (§7) |

## Prérequis

- `torchao>=0.16` installé avant de charger Whosper (sinon échec de chargement sous transformers v5) — §3
- `build_index(lang="wo")` exécuté une fois par runtime neuf : la base Chroma n'est pas versionnée — §4
- Token HF (`HF_TOKEN`) : Kiriku est un dépôt gated, conditions à accepter sur sa page HF
- Token bot Telegram (`TELEGRAM_BOT_TOKEN`) : seul canal pour récupérer un vocal de test depuis un kernel Colab piloté par VS Code — §2

**Objectif J6 (latence, sur Colab GPU uniquement)** — voir §7. Les latences mesurées en CPU local ne sont pas représentatives.

## 0. Setup — détection Colab / local

In [ ]:
import os, sys
from pathlib import Path

try:
    import google.colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    PROJECT_ROOT = Path("/content/noo-far-pipeline")
    if not PROJECT_ROOT.exists():
        !git clone https://github.com/noofar-ia/noo-far-pipeline.git /content/noo-far-pipeline
    else:
        !cd {PROJECT_ROOT} && git pull
else:
    PROJECT_ROOT = Path(r"C:\dev\noo-far-pipeline")

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)   # pour que les chemins relatifs (data/, config/) résolvent
print(f"Projet : {PROJECT_ROOT} | Colab : {ON_COLAB}")

In [ ]:
# Dépendances (Colab surtout ; en local elles sont déjà dans l'env noofar)
if ON_COLAB:
    !pip install chromadb rank-bm25 coqui-tts torchcodec num2words --quiet
    # Shim : isin_mps_friendly retiré en transformers v5, importé par coqui-tts (chemin xTTS)
    import torch, transformers.pytorch_utils as pu
    if not hasattr(pu, "isin_mps_friendly"):
        pu.isin_mps_friendly = lambda e, t: torch.isin(e, t)

In [ ]:
# Token HF — Kiriku est un dépôt gated (conditions à accepter sur sa page HF).
# userdata passe par le canal Colab → indisponible depuis VS Code, d'où les replis.
import os
from huggingface_hub import login

token = os.environ.get("HF_TOKEN")           # 1. variable d'env (posée en début de session VS Code)
if not token and ON_COLAB:
    try:
        from google.colab import userdata     # 2. secret Colab (marche en interface navigateur)
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not token:
    from getpass import getpass               # 3. saisie manuelle (dernier recours)
    token = getpass("HF token : ")

login(token=token)

## 1. Config — bascule wolof

Vérifie que `config.yaml` est bien en `lang: wo`. Sur Colab, force-le au besoin (le dépôt cloné peut être resté en `fr`).

In [ ]:
import yaml
cfg_path = PROJECT_ROOT / "config" / "config.yaml"
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
print("lang        :", cfg["lang"])
print("asr.wo      :", cfg["models"]["asr"]["wo"])
print("tts.wo      :", cfg["models"]["tts"]["wo"])
print("llm.wo      :", cfg["models"]["llm"]["wo"])
print("retrieval.wo:", cfg["rag"]["retrieval"]["wo"])
print("nlu.enabled :", cfg.get("nlu", {}).get("enabled"))
assert cfg["lang"] == "wo", "config.yaml n'est pas en lang: wo"

## 2. Audio d'entrée

Récupération d'un vocal wolof réel via l'API Telegram — seul canal utilisable depuis un kernel Colab piloté par VS Code (`userdata` de Colab n'est accessible qu'en interface navigateur, cf. cellule token HF). Nécessite `TELEGRAM_BOT_TOKEN` dans l'environnement (celui du `.env`, jamais en dur dans le notebook).

Choisir une question **couverte par une fiche wolof**, sinon on ne distingue pas « retrieval raté » de « pas de fiche ».

In [ ]:
import os, requests
from getpass import getpass

TOKEN = os.environ.get("TELEGRAM_BOT_TOKEN") or getpass("Telegram bot token : ")

# getUpdates échoue si un webhook est actif (ex. main.py + ngrok en cours ailleurs) —
# on le désactive temporairement pour pouvoir lire les messages en pull.
requests.get(f"https://api.telegram.org/bot{TOKEN}/deleteWebhook")

In [ ]:
updates = requests.get(f"https://api.telegram.org/bot{TOKEN}/getUpdates").json()

# dernier message contenant un vocal (voice) ou un fichier audio
vocal_updates = [u for u in updates["result"] if "voice" in u["message"] or "audio" in u["message"]]
assert vocal_updates, "aucun vocal trouvé — envoie un message vocal au bot puis relance cette cellule"
dernier = vocal_updates[-1]["message"]
file_id = dernier.get("voice", dernier.get("audio"))["file_id"]
print("file_id :", file_id)

In [ ]:
path = requests.get(f"https://api.telegram.org/bot{TOKEN}/getFile",
                    params={"file_id": file_id}).json()["result"]["file_path"]
url = f"https://api.telegram.org/file/bot{TOKEN}/{path}"
!wget -q -O /content/q01.ogg "{url}"
!ls -la /content/q01.ogg

In [ ]:
import subprocess
from pathlib import Path

AUDIO_IN = Path("/content/q01.ogg")
assert AUDIO_IN.exists(), f"absent : {AUDIO_IN}"

WAV16 = Path("/content/q01.16k.wav")
subprocess.run(["ffmpeg","-y","-i",str(AUDIO_IN),"-ar","16000","-ac","1",str(WAV16)],
               check=True, capture_output=True)
print("prêt :", WAV16)

## 3. ASR — Whosper

**La question de fond du sprint.** Whosper (`CAYTU/whosper-large`, sort en minuscules) transcrit-il correctement le wolof, ou bascule-t-il en français / charabia ? Comparer à ce que tu as réellement dit.

In [ ]:
!pip install -q "torchao>=0.16.0"

In [ ]:
from asr.transcribe import transcribe
texte = transcribe(str(WAV16), lang="wo")
print("ASR :", texte)
question = texte   # la transcription alimente la suite

> Si la sortie francise ou déraille : noter dans le journal des ruptures (maillon ASR). Piste : `pipeline` force peut-être `language="french"` en interne — un `generate_kwargs={"language": "..."}` dans `transcribe.py` peut être nécessaire.

**Libérer la mémoire avant de charger le maillon suivant** (crucial en local 8 Go).

In [ ]:
import gc, torch
from asr.transcribe import _models_cache as _asr_cache
_asr_cache.clear(); gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("ASR déchargé")

## 4. Retrieval — hybride, fiches wolof

La bonne fiche remonte-t-elle à partir de la transcription ? Point de rupture silencieux typique : un `ë`/`ñ` perdu entre l'ASR et ChromaDB fait rater la fiche sans erreur.

In [ ]:
import sys
sys.path.insert(0, "/content/noo-far-pipeline")
from rag.indexer import build_index
build_index(lang="wo")

In [ ]:
from rag.retriever import retrieve
passages = retrieve(question, lang="wo", mode="hybrid")
for doc, meta in passages:
    print(meta.get("source", "?"), "-", doc[:120])

## 5. Génération — extraction directe (LLM reporté S3)

Le premier passage récupéré, tel quel — sans reformulation. C'est ce qui a donné le bout-en-bout fonctionnel (voir §5bis pour pourquoi le LLM est écarté pour l'instant).

In [ ]:
# Extraction directe : la réponse EST le passage le mieux classé (pas de LLM)
reponse = passages[0][0]          # texte de la fiche la plus pertinente
source  = passages[0][1].get("source", "?")
print(f"réponse extraite de : {source}")
print("REP :", reponse[:300])

### 5bis. Essais LLM écartés (pour mémoire)

**Llama 3.2 3B** — testé avec prompt brut et avec chat template : bascule en indonésien sur des questions wolof simples, ou se contente de répéter/régurgiter le prompt. Un seul essai est gardé ci-dessous pour mémoire.

**`soynade-research/Oolel-v0.1`** (candidat nativement wolof) — chargement en 4-bit avec `device_map="auto"` échoue sur ce runtime : `ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM...`. Le modèle (7,6 B) ne tient pas dans la VRAM disponible une fois Whosper et l'embedding model chargés. **Reporté à S3**, à tester sur un runtime dédié plutôt qu'en concurrence avec le reste de la chaîne.

In [ ]:
llm = get_llm("wo")
out = llm([{"role": "user", "content": "Waxal ci wolof: naka nga def?"}],
          max_new_tokens=80, do_sample=False)
print(out[0]["generated_text"][-1]["content"])

In [ ]:
from rag.generator import unload_llm
from asr.transcribe import _models_cache as _asr_cache
import torch, gc

unload_llm()
_asr_cache.clear()
gc.collect(); torch.cuda.empty_cache()
!nvidia-smi --query-gpu=memory.free --format=csv,noheader

## 6. Frontend + TTS — Kiriku

`preparer_pour_tts` verbalise les nombres, applique le lexique, et **force les minuscules pour Kiriku** (son vocab n'a pas de majuscules → noms propres mutilés en silence sinon). Comparer `brut` et `→ TTS` montre ce que le frontend change.

In [ ]:
texte_tts = preparer_pour_tts(reponse, lang="wo", modele="AIHubSN/Kiriku-Wolof-TTS")
audio, sr = synthesize(texte_tts, lang="wo")
print("durée :", round(len(audio)/sr, 2), "s")
display(Audio(audio, rate=sr))

---
## 7. J6 — Chaîne complète et latence (Colab GPU uniquement)

Les cellules ci-dessus testent brique par brique (une en mémoire à la fois). Ci-dessous, la chaîne entière via `process()` — **nécessite d'avoir toute la RAM/VRAM pour les modèles chargés simultanément**, donc Colab, pas les 8 Go locaux.

Les latences par étape (`asr`, `rag_llm`, `frontend`, `tts`) ne sont exploitables **que sur GPU**. En CPU elles sont réelles mais non représentatives. Consigner le matériel avec les chiffres.

In [ ]:
if not ON_COLAB:
    print("⚠ chaîne complète non testée en local (8 Go RAM). Passer sur Colab.")
else:
    from app.pipeline import process
    r = process(str(AUDIO_IN), lang="wo")
    print("ASR :", r["texte_transcrit"])
    print("REP :", r["reponse_texte"])
    print("TTS :", r["texte_tts"])
    print("OUT :", r["audio_out_path"])
    print("LAT :", r["latences"])
    from IPython.display import Audio, display
    import soundfile as sf
    a, s = sf.read(r["audio_out_path"])
    display(Audio(a, rate=s))

## 8. Journal des ruptures

| Question | Maillon | Ce qui a cassé | Friction ou modèle ? |
|---|---|---|---|
| q01 | | | |

**Friction** = corrigible dans le code (encodage, découpage, prompt, frontend). **Modèle** = demande fine-tuning ou changement de modèle. Ne pas confondre : ça décide où investir.